In [16]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

df = pd.read_csv("/Users/toiizumitarou/Documents/project/day5/sample_day2.csv")
df.head()

,patient_id,age,value,date,flag
0,patient_id,age,value,date,flag
1,1,34,5.4,2024-01-01,0
2,2,58,7.1,2024-01-02,1
3,3,45,6.1,2024-01-03,0
4,4,52,7.8,2024-01-04,1


In [20]:
#型確認（念のために）
df.dtypes

patient_id    object
age           object
value         object
date          object
flag          object
dtype: object

In [22]:
# 数値への型変換
df["patient_id"] = pd.to_numeric(df["patient_id"], errors="coerce")
df["age"]        = pd.to_numeric(df["age"],        errors="coerce")
df["value"]      = pd.to_numeric(df["value"],      errors="coerce")
df["flag"]       = pd.to_numeric(df["flag"],       errors="coerce")

# 日付型へ変換
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# 必須列に欠損がある行は落とす
df = df.dropna(subset=["patient_id", "age", "value", "flag", "date"])

# 年齢層カテゴリ
df["age_group"] = pd.cut(
    df["age"],
    bins=[0, 40, 60, 120],
    labels=["young", "middle", "senior"]
)

# high_risk フラグ（検査値 > 7.5 を 1 とみなす）
df["high_risk"] = (df["value"] > 7.5).astype(int)

df.dtypes

/var/folders/by/yd89_rh944vfysllq_zmnfyw0000gn/T/ipykernel_45410/3631918292.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["date"] = pd.to_datetime(df["date"], errors="coerce")


patient_id           float64
age                  float64
value                float64
date          datetime64[ns]
flag                 float64
age_group           category
high_risk              int64
dtype: object

In [24]:
# モデル用データフレーム（カテゴリ列はダミー変数化）
df_model = pd.get_dummies(
    df[["age", "value", "high_risk", "age_group"]],
    drop_first=True
)

X = df_model
y = df["flag"]

X.head(), y.head()

(    age  value  high_risk  age_group_middle  age_group_senior
 1  34.0    5.4          0             False             False
 2  58.0    7.1          0              True             False
 3  45.0    6.1          0              True             False
 4  52.0    7.8          1              True             False
 5  41.0    5.9          0              True             False,
 1    0.0
 2    1.0
 3    0.0
 4    1.0
 5    0.0
 Name: flag, dtype: float64)

In [26]:
#分割
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [28]:
#学習
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [30]:
#評価(accuracy&AUC)
from sklearn.metrics import accuracy_score, roc_auc_score

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(f"Accuracy: {acc:.3f}, AUC: {auc:.3f}")

Accuracy: 0.950, AUC: 1.000
